In [0]:
# Auto Loader: Read CSV files and write to bronze table
# Source: /Volumes/retail_q/volumes/blob_source/transactions_source/
# Target: retail_q.blob_bronze.transactions

# Read CSV files using Auto Loader
df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("cloudFiles.schemaLocation", "/Volumes/retail_q/volumes/blob_source/_schemas/transactions")
  .option("header", "true")
  .option("inferColumnTypes", "true")
  .load("/Volumes/retail_q/volumes/blob_source/transactions_source/")
)

# Write to bronze table with checkpoint for incremental processing
query = (df.writeStream
  .option("checkpointLocation", "/Volumes/retail_q/volumes/blob_source/_checkpoints/transactions")
  .trigger(availableNow=True)
  .toTable("retail_q.blob_bronze.transactions")
)

# Wait for the stream to complete (since we're using availableNow)
query.awaitTermination()

print("✓ Auto Loader stream completed successfully!")

In [0]:
%sql
select count(*) from retail_q.blob_bronze.transactions

In [0]:
# Clear the bronze table and checkpoint to start fresh
spark.sql("TRUNCATE TABLE retail_q.blob_bronze.transactions")

# Remove checkpoint to reset Auto Loader state
dbutils.fs.rm("/Volumes/retail_q/volumes/blob_source/_checkpoints/transactions", True)

print("✓ Bronze table truncated and checkpoint reset")
print("✓ Ready to rerun Cell 1 to load fresh data")